In [1]:
import os
from pathlib import Path

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("Running on Google Colab")
    drive.mount('/content/drive')
    BASE_PATH = Path("/content/drive/MyDrive/Thesis_Final/fake-new-detection")
else:
    print("Running on Local Machine")
    # This notebook lives at <project_root>/notebooks/workflow_coolant_adabelief/
    BASE_PATH = Path().absolute().parents[1]

print(f"IS_COLAB : {IS_COLAB}")
print(f"Base path set to: {BASE_PATH}")

Running on Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IS_COLAB : True
Base path set to: /content/drive/MyDrive/Thesis_Final/fake-new-detection


# Step 2a: Train COOLANT with AdaBelief (Simultaneous)

Train COOLANT with image-caption pairs using AdaBelief optimizer.
- matched (caption_i, image_i) = Real (label 0)
- unmatched (caption_i, image_j) = Fake (label 1)
- **Balanced by construction** - no class weights needed

AdaBelief config: `eps=1e-16, weight_decouple=True, rectify=True`

Prerequisites: Run `1_preprocess_from_pairs.ipynb` first.

- Input: `./processed/coolant_{train,dev,test}.h5`
- Output: `./checkpoints/`

In [2]:
import subprocess, sys
try:
    from adabelief_pytorch import AdaBelief
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "adabelief-pytorch"])
    from adabelief_pytorch import AdaBelief

import os
import math
import random
import warnings
import csv
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support,
    confusion_matrix, accuracy_score, classification_report,
)

warnings.filterwarnings("ignore")

# Redefine project_root relative to BASE_PATH
project_root = BASE_PATH
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root set to: {project_root}")

Project root set to: /content/drive/MyDrive/Thesis_Final/fake-new-detection


In [3]:
# Device and seeds
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Absolute paths — data and outputs live inside the notebook's own folder
_nb_dir = BASE_PATH / "notebooks" / "workflow_coolant_adabelief"
HDF5_DIR = _nb_dir / "processed"
SAVE_DIR = _nb_dir / "checkpoints"
LOG_DIR  = _nb_dir / "logs"

SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device  : {DEVICE}")
print(f"HDF5_DIR: {HDF5_DIR}")
print(f"SAVE_DIR: {SAVE_DIR}")
print(f"LOG_DIR : {LOG_DIR}")

Device  : cuda
HDF5_DIR: /content/drive/MyDrive/Thesis_Final/fake-new-detection/notebooks/workflow_coolant_adabelief/processed
SAVE_DIR: /content/drive/MyDrive/Thesis_Final/fake-new-detection/notebooks/workflow_coolant_adabelief/checkpoints
LOG_DIR : /content/drive/MyDrive/Thesis_Final/fake-new-detection/notebooks/workflow_coolant_adabelief/logs


In [ ]:
# Load data — auto-discover HDF5 files and detect dimensions
import h5py
import glob
from src.processing.coolant import create_coolant_dataloaders

BATCH_SIZE = 32

# Find the HDF5 files: coolant_<image_model>_<text_model>_train.h5
train_candidates = sorted(glob.glob(str(HDF5_DIR / "coolant_*_train.h5")))
if not train_candidates:
    raise FileNotFoundError(f"No coolant_*_tr ain.h5 found in {HDF5_DIR}. Run 1_preprocess_from_pairs.ipynb first.")
if len(train_candidates) > 1:
    print(f"Found multiple feature sets:")
    for c in train_candidates:
        print(f"  {Path(c).name}")
    print(f"Using: {Path(train_candidates[-1]).name}  (latest)")

# Use   the latest (or only) feature set
_train_h5 = train_candidates[-1]
_prefix = Path(_train_h5).name.replace("_train.h5", "")
_dev_h5 = str(HDF5_DIR / f"{_prefix}_dev.h5")
_test_h5 = str(HDF5_DIR / f"{_prefix}_test.h5")

with h5py.File(_train_h5, "r") as f:
    IMAGE_DIM = f["image_features"].shape[1]
    TEXT_EMBED = f["caption_features"].shape[2]
    _image_model = f.attrs.get("image_model", "unknown")
    _text_model = f.attrs.get("text_model", "unknown")

print(f"Features: {_prefix}")
print(f"IMAGE_DIM={IMAGE_DIM} ({_image_model}), TEXT_EMBED={TEXT_EMBED} ({_text_model})")

loaders, datasets = create_coolant_dataloaders(
    train_path=_train_h5, dev_path=_dev_h5, test_path=_test_h5,
    batch_size=BATCH_SIZE,
)

sample_cap, sample_img, _ = datasets["train"][0]
print(f"Caption shape: {sample_cap.shape}, Image shape: {sample_img.shape}")
print(f"Train: {len(loaders['train'])} batches")

Found multiple feature sets:
  coolant_clip_vit_L_14_phobert_base_train.h5
  coolant_clip_vit_L_14_visobert_train.h5
Using: coolant_clip_vit_L_14_visobert_train.h5  (latest)
Features: coolant_clip_vit_L_14_visobert
IMAGE_DIM=1024 (clip-vit-L-14), TEXT_EMBED=768 (visobert)
CoolantPairDataset: 3537 pairs from coolant_clip_vit_L_14_visobert_train.h5
CoolantPairDataset: 1054 pairs from coolant_clip_vit_L_14_visobert_dev.h5
CoolantPairDataset: 876 pairs from coolant_clip_vit_L_14_visobert_test.h5

COOLANT DataLoaders created:
  Train: 111 batches (3537 pairs)
  Dev:   33 batches (1054 pairs)
  Test:  28 batches (876 pairs)
Caption shape: torch.Size([768, 128]), Image shape: torch.Size([1024])
Train: 111 batches


In [5]:
# Config
CONFIG = {
    # Architecture
    "shared_dim": 128,
    "sim_dim": 64,
    "clip_embed_dim": 64,
    "feature_dim": 96,
    "h_dim": 64,
    # AdaBelief optimizer
    "lr": 3e-4,               # 3e-4 (matches stable AdamW baseline; 1e-3 causes mode collapse)
    "eps": 1e-16,
    "betas": (0.9, 0.999),
    "weight_decouple": True,
    "rectify": True,
    "weight_decay": 1e-4,
    # Training
    "max_epochs": 30,
    "patience": 7,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "grad_clip": 1.0,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    # Scheduler: WarmupCosine (primary) + ReduceLROnPlateau (safety net)
    "warmup_fraction": 0.05,
    "min_lr": 1e-6,
    "plateau_factor": 0.5,    # halve LR on plateau
    "plateau_patience": 3,    # wait 3 epochs before reducing
}
print("Config:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

Config:
  shared_dim: 128
  sim_dim: 64
  clip_embed_dim: 64
  feature_dim: 96
  h_dim: 64
  lr: 0.0003
  eps: 1e-16
  betas: (0.9, 0.999)
  weight_decouple: True
  rectify: True
  weight_decay: 0.0001
  max_epochs: 30
  patience: 7
  dropout: 0.3
  label_smoothing: 0.1
  grad_clip: 1.0
  batch_size: 32
  seed: 42
  warmup_fraction: 0.05
  min_lr: 1e-06
  plateau_factor: 0.5
  plateau_patience: 3


In [6]:
# Create and patch model
from src.models.resnet_coolant import (
    PatchedCOOLANT, patch_encoding, patch_clip_projection, patch_cnn_with_dropout,
)

model = PatchedCOOLANT(CONFIG)

patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)
patch_cnn_with_dropout(model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.ambiguity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready: {n_params:,} params")

Model ready: 4,561,712 params


In [7]:
# AdaBelief optimizers (one per module)
def make_adabelief(params):
    return AdaBelief(
        params,
        lr=CONFIG["lr"],
        eps=CONFIG["eps"],
        betas=CONFIG["betas"],
        weight_decouple=CONFIG["weight_decouple"],
        rectify=CONFIG["rectify"],
        weight_decay=CONFIG["weight_decay"],
        print_change_log=False,
    )

optim_sim = make_adabelief(model.similarity_module.parameters())
optim_clip = make_adabelief(model.clip_module.parameters())
optim_det = make_adabelief(model.detection_module.parameters())

print("3 AdaBelief optimizers created")
print(f"  eps={CONFIG['eps']}, weight_decouple={CONFIG['weight_decouple']}, rectify={CONFIG['rectify']}")

Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief
Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief
Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief
3 AdaBelief optimizers created
  eps=1e-16, weight_decouple=True, rectify=True


In [8]:
# Schedulers: WarmupCosine (primary, per-step) + ReduceLROnPlateau (safety, per-epoch)

class WarmupCosineScheduler:
    """Linear warmup then cosine decay. Called per optimizer step."""
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            scale = self.step_count / max(self.warmup_steps, 1)
        else:
            progress = (self.step_count - self.warmup_steps) / max(self.total_steps - self.warmup_steps, 1)
            scale = max(self.min_lr / self.base_lrs[0], 0.5 * (1 + math.cos(math.pi * progress)))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg["lr"] = base_lr * scale

    def get_lr(self):
        return [pg["lr"] for pg in self.optimizer.param_groups]

    def set_base_lr(self, new_base_lr):
        """Called by ReduceLROnPlateau to lower the ceiling."""
        self.base_lrs = [new_base_lr for _ in self.base_lrs]

steps_per_epoch = len(loaders["train"])
total_steps = steps_per_epoch * CONFIG["max_epochs"]
warmup_steps = int(CONFIG["warmup_fraction"] * total_steps)

sch_sim = WarmupCosineScheduler(optim_sim, warmup_steps, total_steps, CONFIG["min_lr"])
sch_clip = WarmupCosineScheduler(optim_clip, warmup_steps, total_steps, CONFIG["min_lr"])
sch_det = WarmupCosineScheduler(optim_det, warmup_steps, total_steps, CONFIG["min_lr"])

# ReduceLROnPlateau: monitors val F1, halves all base LRs if stalled
class PlateauSafetyNet:
    """Monitors a metric. If no improvement for `patience` epochs, halves base LR of all cosine schedulers."""
    def __init__(self, schedulers, factor=0.5, patience=3, min_lr=1e-6):
        self.schedulers = schedulers
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.best = 0.0
        self.wait = 0
        self.n_reductions = 0

    def step(self, metric):
        if metric > self.best:
            self.best = metric
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.wait = 0
                self.n_reductions += 1
                for sch in self.schedulers:
                    old_lr = sch.base_lrs[0]
                    new_lr = max(old_lr * self.factor, self.min_lr)
                    sch.set_base_lr(new_lr)
                print(f"  >> PlateauSafetyNet: reducing base LR {old_lr:.2e} -> {new_lr:.2e} (reduction #{self.n_reductions})")

plateau = PlateauSafetyNet(
    [sch_sim, sch_clip, sch_det],
    factor=CONFIG["plateau_factor"],
    patience=CONFIG["plateau_patience"],
    min_lr=CONFIG["min_lr"],
)

print(f"WarmupCosine: {warmup_steps} warmup / {total_steps} total steps")
print(f"PlateauSafetyNet: factor={CONFIG['plateau_factor']}, patience={CONFIG['plateau_patience']} epochs")

WarmupCosine: 166 warmup / 3330 total steps
PlateauSafetyNet: factor=0.5, patience=3 epochs


In [9]:
# Loss functions - NO class weights (balanced by construction)
from src.processing.coolant.training_utils import make_coolant_pairs, make_detection_batch, soft_cross_entropy

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce_det = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])  # Detection: 2-class
loss_ce_clip = nn.CrossEntropyLoss()  # CLIP contrastive: B-class, no smoothing
loss_kl = nn.KLDivLoss(reduction="batchmean")

print("Loss functions created (separate CE for detection vs CLIP)")

Loss functions created (separate CE for detection vs CLIP)


In [10]:
# Training loop
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tot_loss, tot_n = 0.0, 0
    all_y, all_p = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for caption, image, article_id in tqdm(loader, desc="Train" if train else "Eval"):
            caption = caption.to(DEVICE)
            image = image.to(DEVICE)
            bs = caption.size(0)
            if bs < 4:
                continue

            # Task 1a: Similarity (matched vs unmatched)
            cap_anchor, img_matched, img_unmatched = make_coolant_pairs(caption, image)
            ta_m, ia_m, _ = model.similarity_module(cap_anchor, img_matched)
            ta_u, ia_u, _ = model.similarity_module(cap_anchor, img_unmatched)
            t_cat = torch.cat([ta_m, ta_u])
            i_cat = torch.cat([ia_m, ia_u])
            y_cos = torch.cat([
                torch.ones(ta_m.size(0), device=DEVICE),
                -torch.ones(ta_u.size(0), device=DEVICE),
            ])
            ls = loss_cos(t_cat, i_cat, y_cos)

            if train:
                optim_sim.zero_grad()
                ls.backward()
                torch.nn.utils.clip_grad_norm_(model.similarity_module.parameters(), CONFIG["grad_clip"])
                optim_sim.step()
                sch_sim.step()

            # Task 1b: CLIP contrastive + soft distillation
            cap_pooled = caption.mean(dim=2)
            ie, te = model.clip_module(image, cap_pooled)
            logits = ie @ te.T * math.exp(0.07)
            ids = torch.arange(bs, device=DEVICE)
            ts, is_, _ = model.similarity_module(caption, image)
            soft_m = is_ @ ts.T * math.exp(0.07)
            lc = (loss_ce_clip(logits, ids) + loss_ce_clip(logits.T, ids)) / 2
            ls2 = (soft_cross_entropy(logits, F.softmax(soft_m, 1)) + soft_cross_entropy(logits.T, F.softmax(soft_m.T, 1))) / 2
            l_clip = lc + 0.2 * ls2

            if train:
                optim_clip.zero_grad()
                l_clip.backward()
                torch.nn.utils.clip_grad_norm_(model.clip_module.parameters(), CONFIG["grad_clip"])
                optim_clip.step()
                sch_clip.step()

            # Task 2: Detection (label-smoothed CE + KL)
            det_cap, det_img, det_labels = make_detection_batch(caption, image)
            det_cap_pooled = det_cap.mean(dim=2)
            with torch.no_grad() if not train else torch.enable_grad():
                ie2, te2 = model.clip_module(det_img, det_cap_pooled)
            det_out, attn, skl = model.detection_module(det_cap, det_img, te2, ie2)
            ld = loss_ce_det(det_out, det_labels) + 0.5 * loss_kl(
                F.log_softmax(attn, 1), F.softmax(skl, 1)
            )

            if train:
                optim_det.zero_grad()
                ld.backward()
                torch.nn.utils.clip_grad_norm_(model.detection_module.parameters(), CONFIG["grad_clip"])
                optim_det.step()
                sch_det.step()

            pred = det_out.argmax(1)
            tot_n += det_labels.size(0)
            tot_loss += ld.item() * det_labels.size(0)
            all_y.extend(det_labels.cpu().numpy())
            all_p.extend(pred.cpu().numpy())

    macro_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)
    acc = accuracy_score(all_y, all_p)
    prec, rec, _, _ = precision_recall_fscore_support(all_y, all_p, average=None, zero_division=0)

    return {
        "loss": tot_loss / max(tot_n, 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "real_prec": prec[0] if len(prec) > 0 else 0,
        "real_rec": rec[0] if len(rec) > 0 else 0,
        "fake_prec": prec[1] if len(prec) > 1 else 0,
        "fake_rec": rec[1] if len(rec) > 1 else 0,
    }, all_y, all_p

print("run_epoch defined")

run_epoch defined


In [11]:
# CSV logger setup using LOG_DIR (which is relative to BASE_PATH)
csv_path = LOG_DIR / "train_simultaneous.csv"
csv_fields = ["epoch", "train_loss", "train_acc", "train_f1",
              "val_loss", "val_acc", "val_f1", "val_real_rec", "val_fake_rec", "lr"]

# Open file using the resolved path
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()

print(f"Logging to {csv_path}")

Logging to /content/drive/MyDrive/Thesis_Final/fake-new-detection/notebooks/workflow_coolant_adabelief/logs/train_simultaneous.csv


In [ ]:
# Training
best_val_f1 = 0.0
patience_counter = 0

print(f"Training for max {CONFIG['max_epochs']} epochs (patience={CONFIG['patience']})")
print(f"Optimizer: AdaBelief (lr={CONFIG['lr']}, eps={CONFIG['eps']}, rectify={CONFIG['rectify']})")
print(f"Scheduler: WarmupCosine + PlateauSafetyNet\n")

for epoch in range(1, CONFIG["max_epochs"] + 1):
    train_m, _, _ = run_epoch(loaders["train"], train=True)
    val_m, val_y, val_p = run_epoch(loaders["dev"], train=False)

    # PlateauSafetyNet: check val F1, reduce base LR if stalled
    plateau.step(val_m["macro_f1"])

    current_lr = sch_sim.get_lr()[0]
    cm = confusion_matrix(val_y, val_p)

    print(f"Epoch [{epoch:02d}/{CONFIG['max_epochs']}]  lr={current_lr:.2e}")
    print(f"  Train: loss={train_m['loss']:.4f} acc={train_m['acc']:.4f} F1={train_m['macro_f1']:.4f}")
    print(f"  Val:   loss={val_m['loss']:.4f} acc={val_m['acc']:.4f} F1={val_m['macro_f1']:.4f}")
    print(f"  Val:   real_rec={val_m['real_rec']:.3f} fake_rec={val_m['fake_rec']:.3f}")
    print(f"  CM: {cm.tolist()}")

    # CSV log
    csv_writer.writerow({
        "epoch": epoch, "train_loss": f"{train_m['loss']:.4f}",
        "train_acc": f"{train_m['acc']:.4f}", "train_f1": f"{train_m['macro_f1']:.4f}",
        "val_loss": f"{val_m['loss']:.4f}", "val_acc": f"{val_m['acc']:.4f}",
        "val_f1": f"{val_m['macro_f1']:.4f}",
        "val_real_rec": f"{val_m['real_rec']:.3f}", "val_fake_rec": f"{val_m['fake_rec']:.3f}",
        "lr": f"{current_lr:.2e}",
    })
    csv_file.flush()

    if val_m["macro_f1"] > best_val_f1:
        best_val_f1 = val_m["macro_f1"]
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_f1": best_val_f1,
            "config": CONFIG,
        }, SAVE_DIR / "best_model.pth")
        print(f"  >> Best F1={best_val_f1:.4f}, saved")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            print(f"\nEarly stopping at epoch {epoch}")
            break
    print()

csv_file.close()
print(f"\nDone. Best val F1: {best_val_f1:.4f}")
print(f"Log saved to {csv_path}")

Training for max 30 epochs (patience=7)
Optimizer: AdaBelief (lr=0.0003, eps=1e-16, rectify=True)
Scheduler: WarmupCosine + PlateauSafetyNet



Train:  85%|████████▍ | 94/111 [11:38<02:02,  7.18s/it]

In [ ]:
# Test evaluation
checkpoint = torch.load(SAVE_DIR / "best_model.pth", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded best model from epoch {checkpoint['epoch']} (val F1={checkpoint['val_f1']:.4f})")

test_m, test_y, test_p = run_epoch(loaders["test"], train=False)

print(f"\nTest: loss={test_m['loss']:.4f} acc={test_m['acc']:.4f} F1={test_m['macro_f1']:.4f}")
print(classification_report(test_y, test_p, target_names=["Real(matched)", "Fake(unmatched)"], digits=4))
print(f"Confusion Matrix: {confusion_matrix(test_y, test_p).tolist()}")

# Save test report
report_path = LOG_DIR / "test_report_simultaneous.txt"
with open(report_path, "w") as f:
    f.write(f"Model: COOLANT + AdaBelief (Simultaneous)\n")
    f.write(f"Best epoch: {checkpoint['epoch']}\n")
    f.write(f"Val F1: {checkpoint['val_f1']:.4f}\n")
    f.write(f"Test F1: {test_m['macro_f1']:.4f}\n")
    f.write(f"Test Acc: {test_m['acc']:.4f}\n\n")
    f.write(classification_report(test_y, test_p, target_names=["Real(matched)", "Fake(unmatched)"], digits=4))
    f.write(f"\nConfusion Matrix: {confusion_matrix(test_y, test_p).tolist()}\n")
    f.write(f"\nConfig:\n")
    for k, v in CONFIG.items():
        f.write(f"  {k}: {v}\n")

print(f"Report saved to {report_path}")